# L5 — Fulfilment & demand-state assortment

## Management summary

| Decision | Current model answer |
|---|---|
| Fulfilment cost | **Rs17.61/order = Rs14.25 city + Rs3.36 in-gate** |
| Base topology | **2 runners × 3 gates**; pooling remains conditional upside |
| Local assortment | **Modelled 4,200–8,000 SKUs**, flexed by demand state |
| Network access | **16,500 SKUs** through local stock plus SDFC backfill |
| Financial treatment | **Rs0 incremental assortment saving booked** until pilot evidence opens the gate |

**Management implication:** protect the wide network promise while changing what the campus node holds locally. Use **Runtime → Run all** to refresh the computed summary; the setup and calculation cells are collapsed, and the complete reports remain closed until opened.

In [ ]:
# @title Refresh repository { display-mode: "form" }
from pathlib import Path
import os
import subprocess
import sys

repo_name = "flipkart-wired-x-campus-node"
cwd = Path.cwd()
if (cwd / ".git").exists() and cwd.name == repo_name:
    repo = cwd
else:
    base = Path("/content") if Path("/content").exists() else cwd
    repo = base / repo_name
    if (repo / ".git").exists():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/mba25015-maker/flipkart-wired-x-campus-node.git", str(repo)],
            check=True,
        )

os.chdir(repo)
if os.environ.get("WIRED_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(f"Repository ready: {repo}")

## What the model computes

This notebook reproduces the fulfilment arithmetic behind the adopted delivery cost and topology, then shows how the local assortment changes across Trough, Average, Peak, Exam and Break demand states.

The two decisions remain separate: fulfilment is financially underwritten; assortment is an operating policy whose incremental financial benefit remains unbooked until pilot evidence exists.

In [ ]:
# @title Refresh compact management summary { display-mode: "form" }
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

model_dir = str(Path("Model").resolve())
if model_dir not in sys.path:
    sys.path.insert(0, model_dir)

import sla as fulfilment_model
import fleet_mix as fleet_model
import assortment as assortment_model

cost_legs = fulfilment_model.cost_legs()
runners, runner_share, in_gate_cost, gates = fleet_model.plan_roster()
assortment_plans = assortment_model.plans()
financial_gate = assortment_model.financial_scenario()

display(HTML(f"""
<div style='display:grid;grid-template-columns:repeat(4,minmax(150px,1fr));gap:12px;margin:8px 0 18px'>
  <div style='padding:14px;border:1px solid #d9e2f2;border-radius:10px'><b>Fulfilment cost</b><br><span style='font-size:24px'>Rs{cost_legs['total']:.2f}</span>/order<br><small>Rs{cost_legs['city']:.2f} city + Rs{cost_legs['in_gate']:.2f} in-gate</small></div>
  <div style='padding:14px;border:1px solid #d9e2f2;border-radius:10px'><b>Base topology</b><br><span style='font-size:24px'>{runners} runners</span><br><small>2 per gate × {gates} gates</small></div>
  <div style='padding:14px;border:1px solid #d9e2f2;border-radius:10px'><b>Local range</b><br><span style='font-size:24px'>4,200–8,000</span><br><small>modelled policy by demand state</small></div>
  <div style='padding:14px;border:1px solid #d9e2f2;border-radius:10px'><b>Financial gate</b><br><span style='font-size:24px'>{financial_gate['status']}</span><br><small>Rs0 incremental saving booked</small></div>
</div>
"""))

policy_emphasis = {
    "Trough": "Ambient-led core",
    "Average": "Balanced core",
    "Peak (4x)": "Chilled/RTE, frozen, snacks",
    "Exam night (6x)": "Caffeine, stationery and print",
    "Break": "Ambient core; cold rationalised",
}
summary_rows = []
for policy in assortment_model.POLICIES:
    plan = assortment_plans[policy.key]
    summary_rows.append({
        "Demand state": policy.key,
        "Local SKUs": plan["local_skus"],
        "SDFC backfill": plan["sdfc_tail"],
        "Network access": plan["network_skus"],
        "Cold SKUs": plan["cold_skus"],
        "Policy emphasis": policy_emphasis[policy.key],
    })

summary = pd.DataFrame(summary_rows).set_index("Demand state")
display(summary.style.format({
    "Local SKUs": "{:,.0f}",
    "SDFC backfill": "{:,.0f}",
    "Network access": "{:,.0f}",
    "Cold SKUs": "{:,.0f}",
}))

## How to read the summary

The fulfilment cost is computed from the volume-weighted city gig leg and the fixed-roster in-gate leg. Per-gate staffing is the base case; pooling remains conditional on the pilot establishing cross-gate movement and repricing repositioning.

For assortment, **16,500** is the midpoint of the reported 15,000–18,000-SKU Minutes range. The **4,200–8,000 local caps, category floors and priority weights are modelled operating-policy inputs**, not observed demand forecasts or proven financial savings.

## Full reproducible reports

In [ ]:
# @title Build expandable full reports { display-mode: "form" }
import html as html_lib
import subprocess

def run_report(module_path):
    result = subprocess.run(
        [sys.executable, module_path], capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stdout + result.stderr)
    return result.stdout

reports = [
    ("SLA and batching report", "Model/sla.py"),
    ("Fleet topology and cost report", "Model/fleet_mix.py"),
    ("Demand-state assortment report", "Model/assortment.py"),
]

sections = []
for title, module_path in reports:
    output = html_lib.escape(run_report(module_path))
    sections.append(
        f"<details style='margin:10px 0'><summary style='cursor:pointer;font-weight:600'>{title}</summary>"
        f"<pre style='white-space:pre-wrap;background:#f6f8fa;padding:12px;border-radius:8px'>{output}</pre></details>"
    )
display(HTML("".join(sections)))

## Open the model

- [Fulfilment and SLA source](https://github.com/mba25015-maker/flipkart-wired-x-campus-node/blob/main/Model/sla.py)
- [Fleet-mix source](https://github.com/mba25015-maker/flipkart-wired-x-campus-node/blob/main/Model/fleet_mix.py)
- [Demand-state assortment source](https://github.com/mba25015-maker/flipkart-wired-x-campus-node/blob/main/Model/assortment.py)
- [Public model data](https://github.com/mba25015-maker/flipkart-wired-x-campus-node/tree/main/Model/data)